# Building a Subgraph [Step 07.02 - A compiled graph used as a node]

> **MLCourse - Agentic AI - LangGraph**

The core trick of composition is a single sentence:

> **A compiled LangGraph is a `Runnable`, and `add_node` accepts any `Runnable`.**

That is it. There is no special "subgraph" class. You build a graph, call
`.compile()`, and hand the result to `parent.add_node("name", compiled_child)`.

### What you'll learn

- How to compile a child graph and add it to a parent with `add_node`.
- The **shared-key rule**: direct embedding works only when the schemas overlap.
- How to stream *inside* a subgraph with `stream(..., subgraphs=True)`.
- How subgraph state appears in checkpoints under a **namespace**.
- How to draw the composed graph.

### Key takeaways

- Direct embedding (`add_node("child", compiled)`) requires the parent and child
  to share the state keys the child reads and writes.
- The subgraph runs to completion as one parent "super-step".
- Streaming and checkpointing both understand the nesting - you just have to ask.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                  # environment variable access
import time                                # timing + backoff sleeps
from pathlib import Path                   # locating the track root
from dotenv import load_dotenv             # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we find the track root `03_agentic_ai`,
# then load the (gitignored) .env that lives there. Every provider-touching
# notebook in this track uses exactly this block.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

GROQ_KEY = os.getenv("GROQ_API_KEY")       # never print this value
GROQ_MODEL = "qwen/qwen3.8-27b"            # fast hosted model, generous free tier
OLLAMA_MODEL = "llama3.1:8b"               # local fallback if Groq is unavailable


def make_llm(temperature: float = 0.0, max_tokens: int = 512):
    """Return a chat model. Groq first (fast, hosted); local Ollama as fallback.

    OpenAI is never used anywhere in this course.
    """
    if GROQ_KEY:
        from langchain_groq import ChatGroq
        return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                        temperature=temperature, max_tokens=max_tokens)
    from langchain_ollama import ChatOllama
    return ChatOllama(model=OLLAMA_MODEL, temperature=temperature)


def safe_invoke(model, messages, retries: int = 4, pause: float = 1.5):
    """Invoke a chat model with exponential backoff on rate limits (HTTP 429).

    Groq's free tier allows roughly 8000 tokens per minute. Teaching notebooks
    fire many small calls in a row, so a retry loop is not optional here.
    """
    delay = pause
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(pause)              # pace the next call politely
            return out
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print("  [backoff] %s -- retrying in %.1fs" % (type(exc).__name__, delay))
            time.sleep(delay)
            delay *= 2                     # exponential backoff
    raise RuntimeError("unreachable")


print("Track root :", TRACK.name)
print("Provider   :", "Groq / " + GROQ_MODEL if GROQ_KEY else "Ollama / " + OLLAMA_MODEL)


### 1. Build the child graph first

Our child is a **grader**: given a `draft`, it produces a `grade` and `feedback`.
It has three nodes and one private key (`_word_count`) that the parent should
never care about.

Notice the child is a completely ordinary `StateGraph`. Nothing about it says
"I am a subgraph".

In [2]:
from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END


class GraderState(TypedDict):
    """Child schema. `draft` comes in; `grade` and `feedback` go out."""
    draft: str                 # INPUT  (shared with the parent)
    _word_count: int           # PRIVATE scratch key - parent never sees it
    grade: str                 # OUTPUT (shared with the parent)
    feedback: str              # OUTPUT (shared with the parent)


def measure(state: GraderState) -> dict:
    """Child node 1: cheap deterministic measurement, no LLM needed."""
    return {"_word_count": len(state["draft"].split())}


def judge(state: GraderState) -> dict:
    """Child node 2: ask the model for a one-word grade."""
    prompt = (
        "Grade this draft with exactly ONE word from: excellent, good, weak.\n"
        "Answer with the single word only, nothing else.\n\n"
        "Draft: " + state["draft"]
    )
    word = safe_invoke(grader_llm, prompt).content.strip().lower().split()[0]
    word = word.strip(".,!*")
    if word not in {"excellent", "good", "weak"}:
        word = "good"                      # normalise anything unexpected
    return {"grade": word}


def explain(state: GraderState) -> dict:
    """Child node 3: combine the private measurement with the model's grade."""
    return {"feedback": "Grade '%s' over %d words." % (state["grade"], state["_word_count"])}


grader_llm = make_llm(max_tokens=16)

child = StateGraph(GraderState)
child.add_node("measure", measure)
child.add_node("judge", judge)
child.add_node("explain", explain)
child.add_edge(START, "measure")
child.add_edge("measure", "judge")
child.add_edge("judge", "explain")
child.add_edge("explain", END)

grader_app = child.compile()               # <-- THIS is the reusable value
print("Child compiled:", type(grader_app).__name__)

Child compiled: CompiledStateGraph


### 2. Run the child on its own

This is the first payoff. The child is a standalone runnable, so you can test it
without any parent at all - exactly the "test the middle" problem from notebook 01.

In [3]:
solo = grader_app.invoke({"draft": "LangGraph models agent workflows as a state machine."})
print("grade    :", solo["grade"])
print("feedback :", solo["feedback"])
print("keys out :", sorted(solo.keys()))

grade    : excellent
feedback : Grade 'excellent' over 8 words.
keys out : ['_word_count', 'draft', 'feedback', 'grade']


### 3. Embed it in a parent

The parent writes a `draft`, then the subgraph grades it, then the parent reacts.

For **direct embedding** to work, the parent schema must contain the keys the child
reads and writes: `draft`, `grade`, `feedback`. The child's private `_word_count`
does **not** need to be in the parent - LangGraph will happily let the child use a
key the parent has never heard of, it just won't survive the boundary.

> This "shared keys" requirement is the whole subject of notebook 03. Here we
> satisfy it the easy way: by naming things the same on both sides.

In [4]:
class ParentState(TypedDict):
    topic: str                 # parent-only
    draft: str                 # SHARED with child (in)
    grade: str                 # SHARED with child (out)
    feedback: str              # SHARED with child (out)
    decision: str              # parent-only


def write_draft(state: ParentState) -> dict:
    """Parent node: produce a draft with the model."""
    prompt = ("Write ONE informative sentence about: " + state["topic"] +
              ". Output the sentence only.")
    return {"draft": safe_invoke(writer_llm, prompt).content.strip()}


def decide(state: ParentState) -> dict:
    """Parent node: act on the subgraph's verdict."""
    action = "ship" if state["grade"] in ("excellent", "good") else "rewrite"
    return {"decision": "%s (%s)" % (action, state["feedback"])}


writer_llm = make_llm(max_tokens=90)

parent = StateGraph(ParentState)
parent.add_node("write_draft", write_draft)
parent.add_node("grader", grader_app)      # <<< the compiled child AS A NODE
parent.add_node("decide", decide)
parent.add_edge(START, "write_draft")
parent.add_edge("write_draft", "grader")
parent.add_edge("grader", "decide")
parent.add_edge("decide", END)

parent_app = parent.compile()
out = parent_app.invoke({"topic": "why state machines suit LLM agents"})
print("draft    :", out["draft"])
print("grade    :", out["grade"])
print("decision :", out["decision"])
print()
print("Parent keys:", sorted(out.keys()))
print("'_word_count' leaked into the parent? ", "_word_count" in out)

draft    : State machines provide a structured framework that effectively constrains the non-deterministic nature of LLMs by explicitly defining valid states and transitions, thereby ensuring predictable, reliable, and auditable agent behavior.
grade    : excellent
decision : ship (Grade 'excellent' over 29 words.)

Parent keys: ['decision', 'draft', 'feedback', 'grade', 'topic']
'_word_count' leaked into the parent?  False


> **Read that last line carefully.** The child's private `_word_count` did **not**
> appear in the parent's output. Keys the parent schema does not declare are
> dropped at the boundary. That is the encapsulation you wanted - and also the
> exact mechanism that silently loses data when you get the mapping wrong.

### 4. Visualise the composition

The parent's diagram shows `grader` as a single box. That is the point: the parent
does not know or care that it contains three nodes.

In [5]:
print("PARENT graph:")
print(parent_app.get_graph().draw_ascii())

PARENT graph:
 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
+-------------+  
| write_draft |  
+-------------+  
        *        
        *        
        *        
  +--------+     
  | grader |     
  +--------+     
        *        
        *        
        *        
  +--------+     
  | decide |     
  +--------+     
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


In [6]:
print("PARENT graph expanded one level (xray=1):")
print(parent_app.get_graph(xray=1).draw_ascii())

PARENT graph expanded one level (xray=1):
 +-----------+   
 | __start__ |   
 +-----------+   
        *        
        *        
        *        
+-------------+  
| write_draft |  
+-------------+  
        *        
        *        
        *        
  +---------+    
  | measure |    
  +---------+    
        *        
        *        
        *        
   +-------+     
   | judge |     
   +-------+     
        *        
        *        
        *        
  +---------+    
  | explain |    
  +---------+    
        *        
        *        
        *        
  +--------+     
  | decide |     
  +--------+     
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


### 5. Streaming through the boundary

By default, `parent_app.stream()` treats the subgraph as one opaque step - you see
`grader` finish, but nothing inside it. Pass `subgraphs=True` to see the child's
nodes too. The namespace tuple tells you where you are.

In [7]:
print("--- default stream: subgraph is ONE step ---")
for chunk in parent_app.stream({"topic": "graph checkpoints"}, stream_mode="updates"):
    print("  ", list(chunk.keys()))

--- default stream: subgraph is ONE step ---


   ['write_draft']


   ['grader']
   ['decide']


In [8]:
print("--- stream(subgraphs=True): child nodes are visible ---")
for ns, chunk in parent_app.stream({"topic": "graph checkpoints"},
                                   stream_mode="updates", subgraphs=True):
    where = "PARENT" if not ns else "CHILD " + ns[0]
    print("  %-22s %s" % (where, list(chunk.keys())))

--- stream(subgraphs=True): child nodes are visible ---


  PARENT                 ['write_draft']
  CHILD grader:a26acb84-4acb-7c39-9ede-b435013c0dca ['measure']


  CHILD grader:a26acb84-4acb-7c39-9ede-b435013c0dca ['judge']
  CHILD grader:a26acb84-4acb-7c39-9ede-b435013c0dca ['explain']
  PARENT                 ['grader']
  PARENT                 ['decide']


> **Pitfall:** forgetting `subgraphs=True` is the number-one reason people think
> "my subgraph isn't streaming". It is streaming; you just didn't subscribe to it.
> This builds directly on `05_streaming`.

### 6. Checkpoints are namespaced

Attach a checkpointer to the parent and the child's state is stored under its own
`checkpoint_ns`. You get per-subgraph time travel instead of one giant blob.

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

ck_app = parent.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "sub-demo"}}
ck_app.invoke({"topic": "subgraph namespaces"}, cfg)

print("Parent state keys:", sorted(ck_app.get_state(cfg).values.keys()))
print()
print("Checkpoint history, newest first:")
for st in list(ck_app.get_state_history(cfg))[:5]:
    ns = st.config["configurable"].get("checkpoint_ns", "")
    print("  ns=%-10s next=%s" % (repr(ns), st.next))

Parent state keys: ['decision', 'draft', 'feedback', 'grade', 'topic']

Checkpoint history, newest first:
  ns=''         next=()
  ns=''         next=('decide',)
  ns=''         next=('grader',)
  ns=''         next=('write_draft',)
  ns=''         next=('__start__',)


### 7. Reuse: the same child in a second parent

The child is a value, so a completely different pipeline can use it with zero
changes to the child.

In [10]:
class ReviewState(TypedDict):
    draft: str
    grade: str
    feedback: str
    tag: str


def tag_it(state: ReviewState) -> dict:
    return {"tag": "[%s] %s" % (state["grade"].upper(), state["draft"][:45] + "...")}


second = StateGraph(ReviewState)
second.add_node("grader", grader_app)      # SAME compiled child, different parent
second.add_node("tag_it", tag_it)
second.add_edge(START, "grader")
second.add_edge("grader", "tag_it")
second.add_edge("tag_it", END)

r = second.compile().invoke({"draft": "Agents fail silently when state keys collide."})
print(r["tag"])

[EXCELLENT] Agents fail silently when state keys collide....


### Recap

- `add_node(name, compiled_graph)` is the entire subgraph API.
- Direct embedding requires **shared key names** for everything crossing the boundary;
  unshared keys are silently dropped.
- `stream(..., subgraphs=True)` reveals the child's steps; checkpoints namespace them.
- The child stays reusable because it is just a value.

### Next

**[03_shared_vs_isolated_state](03_shared_vs_isolated_state.ipynb)** - what to do
when the child's schema *doesn't* match the parent's. This is where subgraphs bite.